# CBCT width · optional Colab frontend

The package in `src/cbct_width` is the source of truth. This notebook installs and calls it; it does not generate an independent app file. Use Python 3.12. For clinical limitations and model provenance, read the repository's model card and setup guide.


In [ ]:
from pathlib import Path
import subprocess, sys
assert sys.version_info[:2] == (3, 12), "Select a Python 3.12 runtime for the pinned environment."
REPO = Path('/content/cbct-width')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Basma2753/CBCT-Transverse-Basal-Bone-Width-Analysis-3D-image.git', str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-c', str(REPO / 'requirements-lock.txt'), '-e', str(REPO)], check=True)
print('If numeric libraries were already imported, restart the runtime before continuing.')


## Mount Drive when needed
Skip mounting only when all files and outputs are local. Set persistent paths before importing the batch module.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['CBCT_WORK_DIR'] = '/content/cbct-work'
os.environ['CBCT_MASK_DIR'] = '/content/drive/MyDrive/CBCT_masks'
os.environ['CBCT_CSV_DIR'] = '/content/drive/MyDrive/CBCT_batch_output'


## Optional fresh segmentation

Skip this cell when every case has a valid reusable mask. It uses the runtime's PyTorch installation; verify its CUDA compatibility first. This downloads the separate ~920 MB model archive only when `INSTALL_SEGMENTATION` is enabled. See the upstream terms before reuse/distribution.


In [ ]:
INSTALL_SEGMENTATION = False
if INSTALL_SEGMENTATION:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '/content/cbct-width[segmentation]'], check=True)
    import torch
    print('CUDA available:', torch.cuda.is_available())
    from cbct_width.pipeline import setup_model
    setup_model('/content/cbct-work/nnUNet_results')


## Batch configuration
Change these placeholders. Keep force-rerun off to retain completed results; use the same persistent output directory after restarting a runtime. Do not run concurrent batches into that directory.


In [ ]:
BATCH_ROOT = '/content/drive/MyDrive/CBCT_scans'
BATCH_OUT = '/content/drive/MyDrive/CBCT_results'
REUSE_EXISTING_MASKS = True
FORCE_RERUN = False
SAVE_SEG = True
DEVICE = 'cpu'  # use 'cuda' for a configured GPU inference runtime
FOLD = '5'


In [ ]:
from pathlib import Path
from cbct_width.batch import discover_cases, run_batch
from cbct_width.pipeline import DEFAULT_ARCH_MAPS, get_results_dir
root = Path(BATCH_ROOT)
assert root.is_dir(), f'Scan folder not found: {root}'
cases = discover_cases(root)
assert cases, 'No supported scan volumes found.'
results, landmarks = run_batch(cases, Path(BATCH_OUT), arch_maps=DEFAULT_ARCH_MAPS,
    fold=FOLD, device=DEVICE, results_dir=get_results_dir(), save_seg=SAVE_SEG,
    reuse_existing_masks=REUSE_EXISTING_MASKS, force_rerun=FORCE_RERUN)
results


## Review outputs

Inspect unsuccessful cases, missing widths, relaxed root-count rules, and confidence flags. Mask geometry checks do not prove patient identity. The synthetic suite does not validate your scans or GPU setup. Persistent Drive outputs must be checked after each run.
